<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Crea una Herramienta para Llamar a un Agente

En este laboratorio, explorarás las potentes capacidades de la llamada a herramientas en modelos de lenguaje grandes (LLM) para crear agentes de IA avanzados que puedan interactuar con sistemas externos. Aprenderás a crear herramientas personalizadas que permitan a un LLM realizar acciones específicas, desde extraer identificadores de vídeo hasta obtener transcripciones y metadatos de YouTube. Mediante ejemplos prácticos, primero implementarás la llamada manual a herramientas para comprender su funcionamiento, y luego crearás un sistema de interacción flexible con YouTube capaz de buscar vídeos, extraer transcripciones, obtener contenido de tendencia y generar resúmenes. Al finalizar este laboratorio, comprenderás cómo construir cadenas de llamada a herramientas tanto de secuencia fija como recursivas, lo que permitirá a tus asistentes de IA decidir dinámicamente qué herramientas usar y cuándo usarlas, creando agentes verdaderamente inteligentes que pueden razonar e interactuar con el mundo que les rodea.

## 1. Objetivos

Tras completar este laboratorio, podrás:

- Crear herramientas personalizadas que amplíen las capacidades de los modelos de lenguaje.
- Construir cadenas de llamadas a herramientas, tanto manuales como automatizadas.
- Implementar llamadas recursivas a herramientas para operaciones dinámicas de varios pasos.
- Desarrollar agentes de IA que puedan interactuar programáticamente con el contenido de YouTube.
- Aplicar técnicas de llamadas a herramientas para extraer, procesar y resumir información de fuentes externas.
- Diseñar flujos de trabajo flexibles que permitan a los modelos de lenguaje determinar cuándo y cómo utilizar las herramientas disponibles.

## 2. Configuración


Para este laboratorio, utilizarás las siguientes bibliotecas:

* [`pytube`](https://pytube.io/en/latest/) para acceder programáticamente a vídeos de YouTube y sus metadatos.

* [`youtube-transcript-api`](https://github.com/jdepoix/youtube-transcript-api) para obtener transcripciones de vídeos de YouTube.

* [`langchain`](https://python.langchain.com/docs/get_started/introduction) para crear aplicaciones LLM con herramientas integradas.

* [`langchain-community`](https://python.langchain.com/docs/integrations/providers/) para integraciones adicionales de LangChain.

* [`langchain-openai`](https://python.langchain.com/docs/integrations/llms/openai) para conectarse a los modelos de lenguaje de OpenAI.
* [`yt-dlp`](https://github.com/yt-dlp/yt-dlp) para capacidades mejoradas de extracción de datos de YouTube.

### *2.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *2.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [2]:
import json
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from pytube import YouTube
import re
from typing import List, Dict
import yt_dlp
from IPython.display import display, JSON

# Suprimir advertencias de bibliotecas externas.
import warnings
warnings.filterwarnings("ignore")

# Suprimir advertencias de pytube.
import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

# Suprimir advertencias de yt-dlp.
yt_dpl_logger = logging.getLogger('yt_dlp')
yt_dpl_logger.setLevel(logging.ERROR)

Vamos a inicializar el modelo de lenguaje que impulsará tus capacidades de llamada de herramientas. Este código configura un modelo GPT-4o-mini usando el proveedor OpenAI a través de la interfaz de LangChain, que usarás para procesar consultas y decidir qué herramientas llamar.

In [1]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider= "openai")

### *2.3. Aviso Legal sobre la API*

Este laboratorio utiliza modelos de lenguaje de programación (LLM) proporcionados por OpenAI. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente** fuera del entorno JupyterLab de Skills Network, deberá configurar sus propias claves API. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

Si ejecuta este laboratorio localmente, deberá configurar su propia clave API. Este laboratorio utiliza la función `init_chat_model` de `langchain`. Para usar el modelo, debe establecer la variable de entorno `OPENAI_API_KEY` con su clave API de OpenAI. **NO** ejecute la celda siguiente si no está ejecutando el laboratorio localmente, ya que provocará errores.

In [3]:
# Ignora esta celda si estás ejecutando el laboratorio en el entorno JupyterLab de Skills Network.
# os.environ["OPENAI_API_KEY"] = "your OpenAI API key here"

# 3. Herramientas


### *3.1. Creación de Herramientas Personalizadas con LangChain*

#### 3.1.1. Anatomía de una herramienta

Vamos a proporcionar los componentes básicos de una herramienta. Consideremos la siguiente herramienta:

```python
@tool
def tool_name(parametro_entrada: tipo_entrada) -> tipo_salida:
    """
    Descripción clara de lo que hace la herramienta.

    Argumentos:
    parametro_entrada (tipo_entrada): Descripción de este parámetro

    Devuelve:
    tipo_salida: Descripción del valor devuelto
    """

    # Implementación de la función.
    resultado = process(parametro_entrada)

    return resultado
```

### 3.1.2 Componentes clave

Utilizarás los siguientes componentes clave:

**Decorador @tool**

- Registra la función en LangChain

- Crea atributos de herramienta (.name, .description, .func)

- Genera un esquema JSON para la validación

- Transforma funciones regulares en herramientas invocables

**Nombre de la función**

- Lo utiliza LLM para seleccionar la herramienta adecuada

- Se utiliza como referencia en cadenas y asignaciones de herramientas

- Aparece en los registros de llamadas de la herramienta para la depuración

- Debe indicar claramente el propósito de la herramienta

**Anotaciones de tipo**

- Permite la validación automática de la entrada

- Crea un esquema para los parámetros

- Permite la serialización adecuada de entradas/salidas

- Ayuda a LLM a comprender los formatos de entrada requeridos

**Cadena de documentación**

- Proporciona contexto a LLM para decidir cuándo usar la herramienta

- Documenta los requisitos de los parámetros

- Explica las salidas y el comportamiento esperados

- Es fundamental para la selección de herramientas por parte de LLM

6. **Implementación**

- Ejecuta la operación real

- Gestiona los errores adecuadamente

- Devuelve resultados con el formato correcto

- Debe ser eficiente y robusto

In [5]:
@tool
def extraer_id_video(url: str) -> str:
    """
    Extrae el ID de video de 11 caracteres de una URL de YouTube.

    Args:
        url (str): Una URL de YouTube que contiene un ID de video.

    Returns:
        str: ID de video extraído o mensaje de error si el análisis falla.
    """
    
    # Patrón regex para extraer el ID de video de diferentes formatos de URL de YouTube.
    patron = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(patron, url)

    return match.group(1) if match else "Error: No se pudo extraer el ID de video. Asegúrate de que la URL sea válida."

El decorador envuelve tu función, añadiéndole los atributos (.name, .description, .func) y registrándola en el sistema de herramientas de LangChain. La función original se vuelve accesible mediante el atributo .func, pero el objeto resultante es una instancia de la clase de herramientas de LangChain, con métodos adicionales como `.invoke()` para su invocación directa.

#### 3.1.3. Probando la herramienta de extracción de ID de vídeo

Ahora probarás tu herramienta `extraer_id_video` para verificar que esté registrada correctamente en LangChain. Estas instrucciones de impresión te mostrarán:
1. El nombre de la herramienta (tal como la referenciará el LLM)

2. La descripción de la herramienta (que ayuda al LLM a comprender cuándo usarla)

3. La referencia a la función que se ejecutará

In [6]:
print(extraer_id_video.name)
print("----------------------------")
print(extraer_id_video.description)
print("----------------------------")
print(extraer_id_video.func)

extraer_id_video
----------------------------
Extrae el ID de video de 11 caracteres de una URL de YouTube.

Args:
    url (str): Una URL de YouTube que contiene un ID de video.

Returns:
    str: ID de video extraído o mensaje de error si el análisis falla.
----------------------------
<function extraer_id_video at 0x7c162eabfa60>


#### 3.1.4. Prueba de la herramienta

Aquí, se prueba la ejecución real de la herramienta `extraer_id_video` con una URL real de YouTube. Se puede llamar a la herramienta mediante el método `.invoke()`, una forma práctica de ejecutarla directamente y ver su resultado.

In [13]:
extraer_id_video.invoke("https://www.youtube.com/watch?v=pbXxWaIwyg0")

'pbXxWaIwyg0'

In [9]:
extraer_id_video

StructuredTool(name='extraer_id_video', description='Extrae el ID de video de 11 caracteres de una URL de YouTube.\n\nArgs:\n    url (str): Una URL de YouTube que contiene un ID de video.\n\nReturns:\n    str: ID de video extraído o mensaje de error si el análisis falla.', args_schema=<class 'langchain_core.utils.pydantic.extraer_id_video'>, func=<function extraer_id_video at 0x7c162eabfa60>)

Esta salida muestra que LangChain ha transformado tu función en un objeto `StructuredTool`. Muestra el nombre de la herramienta ('extraer_id_video'), su descripción (nuestra cadena de documentación), un esquema Pydantic para la validación de entrada y una referencia a tu función original.

### *3.2. Lista de Herramientas*

Se crearán varias herramientas para mejorar las capacidades del LLM. Para organizarlas, cree una lista llamada `herramientas`, que es una lista estándar de Python que contiene objetos de herramientas creados con el decorador `@tool`. Esta lista no ejecuta funciones ni determina el orden de las llamadas; simplemente recopila los objetos de herramientas en un solo lugar para que puedan pasarse eficientemente al modelo de lenguaje mediante `llm.bind_tools(herramientas)`. Este enfoque permite que el LLM acceda a todas las herramientas disponibles sin necesidad de registrarlas individualmente.

In [10]:
herramientas = []

herramientas.append(extraer_id_video)

Ahora que ya has comprendido la estructura básica, vamos a definir el resto de las herramientas que necesitarás.

### *3.3. Definición de la Herramienta para Obtener la Transcripción*


Ahora crearás otra herramienta que obtiene la transcripción de un video de YouTube. Esta herramienta utiliza la biblioteca `YouTubeTranscriptApi` para recuperar los subtítulos. Necesitarás el ID del video (que puedes obtener con tu herramienta anterior) y un parámetro de idioma opcional. La función intenta obtener la transcripción y une todos los segmentos de texto en una cadena continua, o devuelve un mensaje de error si no se puede obtener.

In [12]:
from youtube_transcript_api import YouTubeTranscriptApi


@tool
def traer_transcripcion(id_video: str, lenguaje: str = "es") -> str:
    """
    Traer la transcripción de un video de YouTube.
    
    Args:
        id_video (str): El ID del video de YouTube (e.g., "dQw4w9WgXcQ").
        lenguaje (str): Código del idioma para la transcripción (e.g., "en", "es").
    
    Devolver:
        str: La transcripción completa del video o un mensaje de error si no se puede obtener.
    """
    
    try:
        ytt_api = YouTubeTranscriptApi()        
        transcript = ytt_api.fetch(id_video, languages=[lenguaje])

        return " ".join([snippet.text for snippet in transcript.snippets])
    
    except Exception as e:
        return f"Error: {str(e)}"

Vamos a probar la herramienta `traer_trancripcion` llamándola directamente con el método .run() en un ID de vídeo específico. Esto intentará recuperar la transcripción del vídeo con ID "pbXxWaIwyg0" en el idioma inglés predeterminado.

In [14]:
traer_transcripcion.invoke("pbXxWaIwyg0")

'[campana] [música] La brisa la noche y las palmas. Nuestro [música] cuerpos estaban en canal. Mi boca y tu boca no hay una vaina. Están bailando en la cama nuestras almas. [música] Tu cara peosa, tus ojitos de Europa. Estoy loco de verte. Quitarte [música] suave la ropa. Ponle a mi boca esos labios de rosa. Dame romance y hagamos memoria. [música] [música] Tú y yo a las 12 en nuestro cuerpo se conocen. La luna y la noche sacaron foto a [música] nuestra pos envidia que nos tienen ni siquiera nos conocen. Lo hacemos bajo las sábanas para [música] que no vean los dioses. Son shine. Quiero [música] decir bye bye. Tú me quieres guay guay. Tú me tienes fly. Tus ojitos sunshine, tu besitos [música] wi, tú me tienes un trance y lo sabes. La brisa la noche y las palmas. [campana] Nuestros cuerpos estaban en canal. Mi boca y tu boca en una vaina. [música] Oh, están bailando en la cama nuestras almas. Tu cara pecosa, [canto] tus ojitos de Europa. Estoy loco de verte quitarte suave la ropa. Ponle

Agrega la herramienta `traer_transcripcion` a tu lista de herramientas.

In [15]:
herramientas.append(traer_transcripcion)

### *3.4. Definición de la Herramienta de Búsqueda de YouTube*

In [16]:
from pytube import Search
from langchain.tools import tool
from typing import List, Dict

@tool
def busqueda_youtube(consulta: str) -> List[Dict[str, str]]:
    """
    Buscar videos en YouTube que coincidan con la consulta.
    
    Args:
        consulta (str): El término de búsqueda para buscar en YouTube
        
    Devolver:
        Lista de diccionarios que contienen títulos y IDs de videos en formato:
        [{'titulo': 'Video Title', 'id_video': 'abc123'}, ...]
        Devuelve un mensaje de error si la búsqueda falla
    """
    try:
        s = Search(consulta)
        return [
            {
                "titulo": yt.title,
                "id_video": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }

            for yt in s.results
        ]
    
    except Exception as e:
        return f"Error: {str(e)}"

Ahora, probarás tu herramienta `busqueda_youtube` llamándola con el método `.invoke()` y la consulta de búsqueda "Generative AI". Esto devolverá una lista de vídeos de YouTube relacionados con la IA generativa.

In [17]:
salida_busqueda = busqueda_youtube.run("Generative AI")

display(JSON(salida_busqueda))

<IPython.core.display.JSON object>

Se añade la herramienta `busqueda_youtube` a la lista de herramientas.

In [19]:
herramientas.append(busqueda_youtube)

### *3.5. Definición de la Herramienta de Extracción de Metadatos*

Ahora crearás una herramienta que extrae metadatos detallados de un vídeo de YouTube utilizando la biblioteca `yt-dlp`. Esta herramienta toma una URL de YouTube y devuelve información completa sobre el vídeo, incluyendo su título, número de visualizaciones, duración, nombre del canal, número de "me gusta", número de comentarios y marcadores de capítulo.

In [20]:
@tool
def get_toda_metadata(url: str) -> dict:
    """Extraer metadata dado un URL de YouTube, incluyendo título, vistas, duración, canal, me gusta, comentarios y capítulos."""
    
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download= False)

        return {
            'titulo': info.get('title'),
            'vistas': info.get('view_count'),
            'duracion': info.get('duration'),
            'canal': info.get('uploader'),
            'me_gustas': info.get('like_count'),
            'comentarios': info.get('comment_count'),
            'capitulos': info.get('chapters', [])
        }

Ahora, probarás tu herramienta `get_toda_metadata` ejecutándola en una URL específica de un video de YouTube. Esto extraerá información completa sobre el video con ID "pbXxWaIwyg0" sin descargar el contenido del video.

In [21]:
metadata = get_toda_metadata.invoke("https://youtu.be/pbXxWaIwyg0")

display(JSON(metadata))

<IPython.core.display.JSON object>

Agrega la herramienta `get_toda_metadata` a tu lista de herramientas.

In [22]:
herramientas.append(get_toda_metadata)

### *3.6. Definición de la Herramienta de Vídeos de Tendencia*

In [30]:
from langchain.tools import tool
from typing import List, Dict

@tool
def get_videos_tendencia(codigo_region: str) -> List[Dict]:
    """
    Trae los videos de YouTube que están en tendencia para una región específica.
    
    Args:
        codigo_region (str): Código de país de 2 letras (e.g., "US", "IN", "GB")
        
    Devolver:
        Lista de diccionarios con detalles del video: titulo, id_video, canal, cantidad_vistas, duracion
    """

    ydl_opts = {
        'geo_bypass_country': codigo_region.upper(),
        'extract_flat': True,
        'quiet': True,
        'force_generic_extractor': True,
        'logger': yt_dpl_logger
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(
                'https://www.youtube.com/feed/trending',
                download= False
            )
            
            videos_tendecias = []
            for entrada in info['entries']:
                data_video = {
                    'titulo': entrada.get('title', 'N/A'),
                    'id_video': entrada.get('id', 'N/A'),
                    'url': entrada.get('url', 'N/A'),
                    'canal': entrada.get('uploader', 'N/A'),
                    'duracion': entrada.get('duration', 0),
                    'vistas': entrada.get('view_count', 0)
                }

                videos_tendecias.append(data_video)
                
            return videos_tendecias[:25]  # Retorna solo los primeros 25 videos de tendencia.
            
    except Exception as e:
        return [{'error': f"No se pudieron obtener los videos de tendencia: {str(e)}"}]

Ahora, probarás tu herramienta `get_videos_tendencia` ejecutándola con el código de región `"PE"` para obtener videos de tendencia de los Estados Unidos.

In [29]:
videos_tendencia = get_videos_tendencia.invoke("US")

# Mostrar como JSON formateado.
display(JSON(videos_tendencia))

ERROR: [youtube:tab] trending: The channel/playlist does not exist and the URL redirected to youtube.com home page


<IPython.core.display.JSON object>

Ahora, agreguemos la herramienta `get_videos_tendencia` a su lista de herramientas.

In [ ]:
herramientas.append(get_videos_tendencia) # Ya no funciona.

### *3.7. Definición de la Herramienta de Recuperación de Miniaturas*

In [31]:
@tool
def get_miniaturas(url: str) -> List[Dict]:
    """
    Obtiene las miniaturas disponibles para un video de YouTube utilizando su URL.
    
    Args:
        url (str): URL del video de YouTube (cualquier formato)
        
    Devolver:
        Lista de diccionarios con URLs y resoluciones de las miniaturas en el orden nativo de YouTube
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
            info = ydl.extract_info(url, download= False)
            
            miniaturas = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    miniaturas.append({
                        "url": t['url'],
                        "ancho": t.get('width'),
                        "alto": t.get('height'),
                        "resolucion": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return miniaturas

    except Exception as e:
        return [{"error": f"Error al obtener miniaturas: {str(e)}"}]

Ahora, probarás tu herramienta `get_miniaturas` ejecutándola en una URL de vídeo específica de YouTube. Esto extraerá información sobre todas las imágenes en miniatura disponibles para el vídeo.

In [ ]:
miniaturas = get_miniaturas.invoke("https://www.youtube.com/watch?v=pbXxWaIwyg0")

display(JSON(miniaturas))

<IPython.core.display.JSON object>

Ahora, agreguemos la herramienta `get_miniaturas` a su lista de herramientas.

In [34]:
herramientas.append(get_miniaturas)

##  4. Vinculación de Herramientas


Ahora, vincularás tu colección de herramientas al modelo de lenguaje. Esto permite que el modelo acceda y utilice tus herramientas personalizadas de YouTube durante las conversaciones. Al vincular las herramientas, le das al modelo la capacidad de llamar a estas funciones cuando determine que son necesarias para atender una solicitud del usuario, lo que permite que el modelo conozca las capacidades de tus herramientas y cómo utilizarlas.

In [35]:
llm_con_herramientas = llm.bind_tools(herramientas)

La función ```bind_tools()``` pasa toda esta información al modelo de lenguaje. Convierte los atributos de cada herramienta (nombre, descripción, esquema de parámetros) en un formato estandarizado que el LLM puede comprender y utilizar para determinar cuándo y cómo llamar a herramientas específicas en función de las solicitudes del usuario. Similar al siguiente código donde se almacena el esquema de cada herramienta:

In [36]:
for herramienta in herramientas:
    schema = {
   "nombre": herramienta.name,
   "descripcion": herramienta.description,
   "parametros": herramienta.args_schema.schema() if herramienta.args_schema else {},
   "retorno": herramienta.return_type if hasattr(herramienta, "return_type") else None}
    
    display(JSON(schema))
    

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

### *4.1. Cómo el LLM Llama a una Herramienta*

Ahora, defina una consulta de usuario de ejemplo que solicite un resumen de un video específico de YouTube. Esta consulta se utilizará para demostrar cómo su LLM puede comprender una solicitud en lenguaje natural y usar las herramientas adecuadas que usted ha proporcionado para satisfacerla.

In [75]:
consulta = "Quiero el resumen del video de YouTube: https://www.youtube.com/watch?v=pbXxWaIwyg0 en español"

print(consulta)

Quiero el resumen del video de YouTube: https://www.youtube.com/watch?v=pbXxWaIwyg0 en español


Repetir un objeto de mensaje para representar la consulta del usuario. Envolverás la cadena de consulta en un objeto HumanMessage, que es la forma estándar de formatear las entradas del usuario en LangChain. Representa un mensaje humano, ya que se espera que una persona inicie la interacción.

In [76]:
mensajes = [HumanMessage(content = consulta)]

print(mensajes)

[HumanMessage(content='Quiero el resumen del video de YouTube: https://www.youtube.com/watch?v=pbXxWaIwyg0 en español', additional_kwargs={}, response_metadata={})]


### *4.2. Proceso de Vinculación de Herramientas de LangChain*

In [77]:
respuesta_1 = llm_con_herramientas.invoke(mensajes)
respuesta_1

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 443, 'total_tokens': 473, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dac1f01875', 'id': 'chatcmpl-De8I3nEHIXlj70ghSkU9enLkwhXuI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1451-b341-7622-9d98-dc0d5e4dab9b-0', tool_calls=[{'name': 'extraer_id_video', 'args': {'url': 'https://www.youtube.com/watch?v=pbXxWaIwyg0'}, 'id': 'call_dW7nllA1SIyLVaQjmfB6qi9y', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 443, 'output_tokens': 30, 'total_tokens': 473, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

Agrega la respuesta del modelo de lenguaje a tu historial de conversación. Tras recibir la respuesta del modelo de lenguaje (que contiene la llamada a la herramienta para extraer el ID del video), agrégala a tu lista de mensajes para mantener el contexto de la conversación. Esto crea el historial de chat que se utilizará para interacciones posteriores con el modelo.

In [78]:
mensajes.append(respuesta_1)

### *4.3. Extracción de Información de Llamadas a Herramientas*

Tras recibir la respuesta del LLM, es necesario extraer la información estructurada de las llamadas a herramientas. La línea `llamar_herramienta_1 = respuesta_1.tool_calls` obtiene los objetos de llamada a herramientas que contienen la herramienta que el LLM ha decidido usar y los parámetros que debe pasarle. Esta información se utilizará para ejecutar la herramienta adecuada con las entradas correctas.

In [79]:
mapeo_herramientas = {
    "get_miniaturas" : get_miniaturas,    
    "extraer_id_video": extraer_id_video,
    "traer_transcripcion": traer_transcripcion,
    "busqueda_youtube": busqueda_youtube,
    "get_toda_metadata": get_toda_metadata
}

Extracción de las llamadas a herramientas de la respuesta del modelo de lenguaje. Cuando el modelo de lenguaje determina que necesita usar una de tus herramientas, incluye llamadas a herramientas estructuradas en su respuesta. Aquí, accedes a esas llamadas para ver qué herramientas decidió usar el modelo para completar la solicitud de resumir el video de YouTube.

In [80]:
llamar_herramienta_1 = respuesta_1.tool_calls

llamar_herramienta_1

[{'name': 'extraer_id_video',
  'args': {'url': 'https://www.youtube.com/watch?v=pbXxWaIwyg0'},
  'id': 'call_dW7nllA1SIyLVaQjmfB6qi9y',
  'type': 'tool_call'}]

Aquí se muestra la estructura de la llamada a la herramienta que el LLM decidió realizar. La llamada tiene formato de diccionario con los siguientes componentes clave:

1. `name`: 'extraer_id_video' - Identifica la herramienta que el LLM desea usar primero (la herramienta de extracción de ID de video).

2. `args`: Contiene los argumentos que se le pasan a la herramienta; en este caso, la URL de YouTube de la consulta.

3. `id`: Un identificador único para esta llamada a la herramienta, que ayuda a rastrear el par solicitud/respuesta.

4. `type`: Indica que se trata de una llamada a una herramienta y no de otro tipo de respuesta de IA.

Esto demuestra que el LLM comprendió correctamente que primero debe extraer el ID del video de la URL antes de poder resumir el contenido del video.

Acceder al nombre de la primera herramienta que el LLM decidió usar. Aquí se extrae solo el componente de nombre `('extraer_id_video')` de la primera llamada a la herramienta en la lista.

In [81]:
nombre_herramienta = llamar_herramienta_1[0]['name']
print(nombre_herramienta)

extraer_id_video


Necesitas un ID de herramienta para ayudar al LLM a saber de dónde proviene el resultado:

In [82]:
id_llamada_herramienta = llamar_herramienta_1[0]['id']
print(id_llamada_herramienta)

call_dW7nllA1SIyLVaQjmfB6qi9y


Acceso a los argumentos que deben pasarse a la herramienta seleccionada. Aquí, se extrae el componente de argumentos de la primera llamada a la herramienta, que contiene la URL de YouTube que debe procesarse.

In [83]:
args = llamar_herramienta_1[0]['args']
print(args)

{'url': 'https://www.youtube.com/watch?v=pbXxWaIwyg0'}


Agregar la respuesta del modelo de lenguaje a tu historial de conversación. Tras recibir la respuesta del modelo de lenguaje (que contiene la llamada a la herramienta para extraer el ID del video), la agregas a tu lista de mensajes para mantener el contexto de la conversación. Esto crea el historial de chat que se utilizará para interacciones posteriores con el modelo.

Ejecutando la llamada a la herramienta solicitada por LLM. Aquí, se utiliza el diccionario de mapeo de herramientas para:

1. Buscar la función apropiada según el nombre de la herramienta ('extraer_id_video').

2. Llamar a esa función con los argumentos proporcionados por LLM.

3. Capturar el resultado (el ID del video extraído).

Esto muestra cómo se pueden ejecutar programáticamente las herramientas que LLM decidió usar. Primero, se obtiene la herramienta de `mapeo_herramientas`.

In [84]:
mi_herramienta = mapeo_herramientas[llamar_herramienta_1[0]['name']]

A continuación, llamarás a la herramienta con los siguientes argumentos:

In [85]:
id_video = mi_herramienta.invoke(llamar_herramienta_1[0]['args'])
id_video

'pbXxWaIwyg0'

Agregar el resultado de la herramienta al historial de la conversación. Crearás un `ToolMessage` que contiene:

1. El resultado de la ejecución de la herramienta (el ID del video extraído)

2. El ID de la llamada original a la herramienta para vincular esta respuesta con la solicitud específica.

Al agregar este mensaje al historial de la conversación, informas al LLM sobre los resultados de la ejecución de la herramienta, que podrá utilizar en su próxima respuesta.

In [86]:
mensajes.append(ToolMessage(content = id_video, tool_call_id = llamar_herramienta_1[0]['id']))

Envía tu conversación actualizada al LLM y guarda su nueva respuesta. Ahora que has informado al modelo sobre el ID de vídeo extraído, invócalo de nuevo para continuar el proceso. El modelo verá tanto la consulta original como el resultado de la extracción del ID de vídeo, lo que le permitirá determinar el siguiente paso necesario para resumir el vídeo de YouTube.

In [87]:
respuesta_2 = llm_con_herramientas.invoke(mensajes)
respuesta_2

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 490, 'total_tokens': 519, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dac1f01875', 'id': 'chatcmpl-De8IJ2Lr3fqO1tFBjhdhQdKAid1sB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1451-f281-76f3-8f3e-fddae53c44b5-0', tool_calls=[{'name': 'traer_transcripcion', 'args': {'id_video': 'pbXxWaIwyg0', 'lenguaje': 'es'}, 'id': 'call_DHEfvo0rZGpY603rf2tMEv4t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 490, 'output_tokens': 29, 'total_tokens': 519, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

El resultado es un mensaje de IA. Envía tu conversación actualizada al LLM y guarda su nueva respuesta. Ahora que has informado al modelo sobre el ID del video extraído, lo invocarás de nuevo para continuar el proceso. El modelo verá tanto la consulta original como el resultado de la extracción del ID del video, lo que le permitirá determinar el siguiente paso necesario para resumir el video de YouTube.

In [88]:
mensajes.append(respuesta_2)

Extrayendo las llamadas a la herramienta de la segunda respuesta del modelo de lenguaje. Tras recibir el ID del vídeo, es probable que el modelo de lenguaje decida utilizar otra herramienta para ayudar con la tarea de resumen.

In [89]:
llamar_herramienta_2 = respuesta_2.tool_calls
llamar_herramienta_2

[{'name': 'traer_transcripcion',
  'args': {'id_video': 'pbXxWaIwyg0', 'lenguaje': 'es'},
  'id': 'call_DHEfvo0rZGpY603rf2tMEv4t',
  'type': 'tool_call'}]

Aquí se puede observar que el modelo LLM ha decidido utilizar la herramienta `traer_transcripcion` como siguiente paso.

El modelo pasa dos argumentos a la herramienta de obtención de transcripciones:
1. `id_video`: 'T-D1OfcDW1M' - El ID extraído de la URL original de YouTube.

2. `lenguaje`: 'es' - Solicita la transcripción en español, tal como se especificó en la consulta del usuario.

Obtención de la transcripción mediante el ID de vídeo obtenido en el paso anterior. Aquí, se ejecuta la segunda herramienta solicitada por LLM:

1. Búsqueda de la función `('traer_transcripcion')` correspondiente en el mapa de herramientas.

2. Invocación de la función con el ID de vídeo y los parámetros de idioma.

3. Almacenamiento del contenido de la transcripción resultante.


In [90]:
traer_transcripcion_herramienta = mapeo_herramientas[llamar_herramienta_2[0]['name']].invoke(llamar_herramienta_2[0]['args'])
traer_transcripcion_herramienta

'[campana] [música] La brisa la noche y las palmas. Nuestro [música] cuerpos estaban en canal. Mi boca y tu boca no hay una vaina. Están bailando en la cama nuestras almas. [música] Tu cara peosa, tus ojitos de Europa. Estoy loco de verte. Quitarte [música] suave la ropa. Ponle a mi boca esos labios de rosa. Dame romance y hagamos memoria. [música] [música] Tú y yo a las 12 en nuestro cuerpo se conocen. La luna y la noche sacaron foto a [música] nuestra pos envidia que nos tienen ni siquiera nos conocen. Lo hacemos bajo las sábanas para [música] que no vean los dioses. Son shine. Quiero [música] decir bye bye. Tú me quieres guay guay. Tú me tienes fly. Tus ojitos sunshine, tu besitos [música] wi, tú me tienes un trance y lo sabes. La brisa la noche y las palmas. [campana] Nuestros cuerpos estaban en canal. Mi boca y tu boca en una vaina. [música] Oh, están bailando en la cama nuestras almas. Tu cara pecosa, [canto] tus ojitos de Europa. Estoy loco de verte quitarte suave la ropa. Ponle

Para añadir el contenido de la transcripción al historial de la conversación, crea otro `ToolMessage` que contiene el texto de la transcripción y el ID de la llamada a la herramienta que la solicitó. Esto permite que LLM acceda al contenido del vídeo para generar un resumen.

In [91]:
mensajes.append(ToolMessage(content = traer_transcripcion_herramienta, tool_call_id = llamar_herramienta_2[0]['id']))

Genera el resumen final enviando el historial completo de la conversación al LLM. Ahora que el modelo tiene acceso tanto al ID del video como a la transcripción completa, lo invocarás una vez más para generar el resumen que solicitó el usuario.

In [92]:
mensajes

[HumanMessage(content='Quiero el resumen del video de YouTube: https://www.youtube.com/watch?v=pbXxWaIwyg0 en español', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 443, 'total_tokens': 473, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dac1f01875', 'id': 'chatcmpl-De8I3nEHIXlj70ghSkU9enLkwhXuI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1451-b341-7622-9d98-dc0d5e4dab9b-0', tool_calls=[{'name': 'extraer_id_video', 'args': {'url': 'https://www.youtube.com/watch?v=pbXxWaIwyg0'}, 'id': 'call_dW7nllA1SIyLVaQjmfB6qi9y', 'type': 'tool_call'}], invalid_tool_c

In [ ]:
resumen = llm_con_herramientas.invoke(mensajes)

In [ ]:
resumen

AIMessage(content='El video contiene una canción que habla sobre el romance y la conexión entre dos personas. A lo largo de la letra, se menciona la sensación de una noche especial, donde la brisa, el baile y las miradas entre los protagonistas crean un ambiente íntimo. Se describen momentos de deseo, con referencias a besos y caricias, y se expresa la locura de estar juntos y disfrutar de la compañía del otro.\n\nSe destacan elementos visuales como la luna y las palmas, que añaden un toque romántico a la atmósfera. La letra también menciona la exclusividad de su relación, sugiriendo que lo que tienen es algo único y que las miradas ajenas son irrelevantes para ellos.\n\nEn resumen, el video captura momentos de pasión, afecto y una conexión profunda entre dos almas que se encuentran y disfrutan de la intimidad juntos.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 848, 'total_tokens': 1029, 'completion_tokens_details

In [95]:
# Definir los pasos del procesamiento
def ejecutar_herramienta(llamar_herramienta):
    """Ejecutar una sola llamada a la herramienta y devolver un ToolMessage"""
    try:
        resultado = mapeo_herramientas[llamar_herramienta["name"]].invoke(llamar_herramienta["args"])

        return ToolMessage(
            content= str(resultado),
            tool_call_id= llamar_herramienta["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id= llamar_herramienta["id"]
        )

        

Ahora vas a encadenar todas tus funciones o herramientas, pero antes debes formatear los datos correctamente. No solo necesitas almacenar la salida de cada herramienta, sino también información de estado, como los ID de las herramientas. Para hacerlo de forma eficaz, debes asegurarte de que la salida de cada herramienta se pueda pasar correctamente al siguiente paso de tu canalización. El componente RunnablePassthrough te permite mantener el estado a lo largo de la cadena, añadiendo o transformando datos en cada paso, lo que lo hace ideal para conectar tus diversas herramientas en un flujo de trabajo coherente.

La función RunnableLambda, ubicada al final de la cadena, tiene una función diferente: extrae solo el resultado final que deseas presentar al usuario. Después de todas las llamadas a las herramientas y el procesamiento de mensajes, tienes un objeto de estado completo con muchos campos, pero el usuario normalmente solo necesita la respuesta final. RunnableLambda transforma este estado completo en la información que deseas devolver.

In [96]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

## 5. Construyendo la Cadena de Resumen

Ahora, combinarás tus funciones en una cadena de resumen completa usando el operador de tubería `|`, que aplica las funciones secuencialmente (de forma similar a la composición de funciones, donde `f|g(x)` es equivalente a `f(g(x))`).

El flujo de trabajo sigue estos pasos:
1. Convertir la solicitud de entrada a un HumanMessage.

2. Pasar el mensaje a LLM con herramientas.

3. Extraer las llamadas a las herramientas de la respuesta de LLM.

4. Actualizar el historial de mensajes con los resultados de las herramientas.

5. Enviar los mensajes actualizados de vuelta a LLM.

6. Repetir los pasos 3 a 5 según sea necesario.

7. Finalmente, extraer solo el contenido del mensaje final usando RunnableLambda.

Cada paso mantiene el estado usando RunnablePassthrough hasta llegar al mensaje final, momento en el que aplicarás RunnableLambda para extraer solo el texto del resumen.

In [106]:
cadena_resumen_video = (
    # Comienza con el mensaje del usuario.
    RunnablePassthrough.assign(
        mensajes= lambda x: [HumanMessage(content= x["consulta"])]
    )
    # Primero llamada al LLM (extraer ID del video).
    | RunnablePassthrough.assign(
        respuesta_ia= lambda x: llm_con_herramientas.invoke(x["mensajes"])
    )
    # Proceso de la primera llamada a la herramienta.
    | RunnablePassthrough.assign(
        mensaje_herramienta= lambda x: [
            ejecutar_herramienta(tc) for tc in x["respuesta_ia"].tool_calls
        ]
    )
    # Actualización del historial de mensajes.
    | RunnablePassthrough.assign(
        mensajes= lambda x: x["mensajes"] + [x["respuesta_ia"]] + x["mensaje_herramienta"]
    )

    # Segunda llamada al LLM (obtener transcripción).
    | RunnablePassthrough.assign(
        respuesta_ia2= lambda x: llm_con_herramientas.invoke(x["mensajes"])
    )        
    # Procesar segunda llamada a la herramienta.
    | RunnablePassthrough.assign(
        mensaje_herramienta2= lambda x: [
            ejecutar_herramienta(tc) for tc in x["respuesta_ia2"].tool_calls
        ]
    )
    # Actualización final del historial de mensajes.
    | RunnablePassthrough.assign(
        mensajes= lambda x: x["mensajes"] + [x["respuesta_ia2"]] + x["mensaje_herramienta2"]
    )
    # Resumen final con el LLM, utilizando todo el historial de mensajes actualizado.
    | RunnablePassthrough.assign(
        resumen= lambda x: llm_con_herramientas.invoke(x["mensajes"])
    )
    # Retornar solo el texto del resumen.
    | RunnableLambda(lambda x: x["resumen"].content)
)


Así es como se invoca la cadena de resumen con la URL de un video de YouTube: esta envía la consulta con la URL a la cadena, que extrae automáticamente el ID del video, obtiene la transcripción y genera un resumen del contenido.

**Nota: Si encuentra algún problema con el enlace de video proporcionado a continuación, pruebe con cualquier otro enlace de video de YouTube.**

In [131]:
# Usando la cadena de resumen de video con una consulta que incluye una URL de YouTube, solicitando el resumen en español.
resultado = cadena_resumen_video.invoke({
    "consulta": "Resumen este video de YouTube: https://www.youtube.com/watch?v=pbXxWaIwyg0"
})

print("Resumen del video:\n", resultado)

Resumen del video:
 El video de YouTube presenta una canción con letras que evoca una atmósfera romántica y sensual. La temática gira en torno a una conexión íntima entre dos personas, destacando momentos de cercanía, baile y deseo. Se menciona la brisa, la luna y el ambiente nocturno, creando una imagen poética de su relación.

Las letras hacen referencia a la belleza y atractivo físico de la pareja, mencionando sus ojos y labios, y sugiriendo un deseo de desnudarse emocional y físicamente. La repetición de frases como "están bailando en la cama nuestras almas" resalta la conexión profunda que tienen.

En esencia, el video captura la esencia de un romance apasionado, lleno de memorias y momentos compartidos bajo la luz de la luna.


Hasta ahora, has demostrado cómo orquestar manualmente el proceso de llamada a herramientas paso a paso. Primero, invocaste el LLM con la consulta del usuario, interpretaste su decisión de usar la herramienta `extraer_id_video`, la ejecutaste, enviaste el resultado de vuelta al LLM, procesaste su siguiente decisión de usar la herramienta `traer_transcripcion`, la ejecutaste y, finalmente, hiciste que el LLM generara un resumen basado en la transcripción.

Ahora verás cómo lograr el mismo flujo de trabajo de manera más eficiente usando la funcionalidad de encadenamiento de LangChain, que automatiza este proceso de selección, ejecución y manejo de respuestas de las herramientas.

### *5.1. Creación de la Configuración Inicial del Mensaje*

Aquí se configura el primer paso de la cadena que gestionará la consulta inicial del usuario. El método `RunnablePassthrough.assign` crea un componente que toma un diccionario de entrada que contiene una "consulta" y lo convierte en una lista que contiene un único objeto `HumanMessage`.

In [111]:
configuracion_inicial = RunnablePassthrough.assign(
    mensajes= lambda x: [HumanMessage(content= x["consulta"])]
)

### *5.2. Definición de la Primera Interacción con el Modelo de Lenguaje*

Aquí, crearás el segundo paso de tu cadena, que gestiona la primera interacción con el modelo de lenguaje. Este componente toma los mensajes formateados del paso anterior, los envía a tu modelo de lenguaje integrado y captura la respuesta en un campo llamado "respuesta_ia".

In [112]:
primera_llamada_llm = RunnablePassthrough.assign(
    respuesta_ia= lambda x: llm_con_herramientas.invoke(x["mensajes"])
)

### *5.3. Procesamiento de la Primera Llamada a la Herramienta*

Aquí se define el paso de procesamiento que gestiona la primera llamada a la herramienta del LLM. Este componente:
1. Ejecuta cada llamada a la herramienta pasándola a la función `ejecutar_herramienta`, que ejecuta la herramienta correspondiente y devuelve el resultado como un `ToolMessage`.

2. Actualiza el historial de mensajes combinando los mensajes originales, la respuesta del LLM, las llamadas a la herramienta y los resultados de la misma.

3. Prepara el estado de conversación actualizado para la siguiente interacción con el LLM.

In [113]:
primer_proceso_herramienta = RunnablePassthrough.assign(
    mensajes_herramienta= lambda x: [
        ejecutar_herramienta(tc) for tc in x["respuesta_ia"].tool_calls
    ]
).assign(
    mensajes= lambda x: x["mensajes"] + [x["respuesta_ia"]] + x["mensajes_herramienta"]
)

### *5.4. Definición de la Segunda Interacción con el Modelo de Lenguaje*

Aquí, se crea el siguiente paso en la cadena que gestiona la segunda interacción con el modelo de lenguaje. Este componente toma el historial de mensajes actualizado (que ahora incluye los resultados de la primera llamada a la herramienta) y lo envía nuevamente al modelo de lenguaje.

In [114]:
segunda_llamada_llm = RunnablePassthrough.assign(
    respuesta_ia2= lambda x: llm_con_herramientas.invoke(x["mensajes"])
)

### *5.5. Procesamiento de la Segunda Llamada a la Herramienta*

Aquí se define el paso de procesamiento que gestiona la segunda llamada a la herramienta LLM. Al igual que en el primer paso de procesamiento, este componente ejecuta las llamadas a la herramienta (normalmente, recuperando la transcripción), crea mensajes con los resultados y actualiza el historial de mensajes combinando toda la información para el paso final de resumen.

In [127]:
segundo_proceso_herramienta = RunnablePassthrough.assign(
    mensajes_herramienta2= lambda x: [
        ejecutar_herramienta(tc) for tc in x["respuesta_ia2"].tool_calls
    ]
).assign(
    mensajes= lambda x: x["mensajes"] + [x["respuesta_ia2"]] + x["mensajes_herramienta2"]
)

### *5.6. Generación del Resumen Final*

Aquí se define el paso final que genera el resumen del video de YouTube. Este componente:
1. Recibe el historial completo del mensaje (que ahora incluye la consulta original, las llamadas a la herramienta y los resultados de la misma).

2. Invoca a LLM una última vez para generar el resumen.

3. Extrae únicamente el campo de contenido de la respuesta de LLM.

4. Utiliza una función RunnableLambda para devolver solo el texto del resumen como resultado final.

In [128]:
resumen_final = RunnablePassthrough.assign(
    resumen= lambda x: llm_con_herramientas.invoke(x["mensajes"])
) | RunnableLambda(lambda x: x["resumen"].content)

### *5.7. Ensamblando la Cadena Completa*

Ahora, combinarás todos los componentes individuales que has definido en una única cadena coherente. Al conectar cada paso con el siguiente, crearás un flujo de trabajo que:
1. Formatea la consulta inicial

2. Obtiene la primera respuesta de LLM (extracción del ID del vídeo)

3. Procesa la primera llamada a la herramienta

4. Obtiene la segunda respuesta de LLM (solicitud de transcripción)

5. Procesa la segunda llamada a la herramienta

6. Genera el resumen final

In [129]:
cadena = (
    configuracion_inicial
    | primera_llamada_llm
    | primer_proceso_herramienta
    | segunda_llamada_llm
    | segundo_proceso_herramienta
    | resumen_final
)

Ahora, estás probando tu cadena automatizada con la consulta de resumen de vídeo original que gestionaste manualmente antes. Al pasar la misma consulta a tu cadena, puedes confirmar que produce los mismos resultados, pero de una manera mucho más optimizada.

In [130]:
consulta = {"consulta": "Quiero resumir el video de youtube: https://www.youtube.com/watch?v=awybIugZyw8 en español"}
resultado = cadena.invoke(consulta)

print("Resumen del video:\n", resultado)

Resumen del video:
 El video es una canción que habla de la angustia y la espera en una relación amorosa. El cantante expresa su deseo de que su pareja regrese, cuestionando cuánto tiempo tendrá que esperar y lamentando la distancia que existe entre ellos. 

A lo largo de la letra, se siente un profundo anhelo por la cercanía de su pareja, pero también una frustración al sentir que su amor no es correspondido adecuadamente. Se menciona la dificultad de vivir en una situación donde el amor no se puede expresar plenamente y se siente atrapado en una red emocional.

El tema central gira en torno a la necesidad de claridad en la relación y el dolor de esperar a alguien que parece estar distante o no disponible. La canción resalta el deseo de sanar las heridas y la lucha por encontrar una salida en medio del sufrimiento emocional.


## 6. Flujo de Cadena Recursiva

Ahora que has creado una cadena que funciona bien para tu proceso específico de llamada a herramientas en dos pasos, debes considerar escenarios más complejos. Tu cadena actual se limita a exactamente dos llamadas a herramientas en una secuencia fija. En aplicaciones reales, podrías necesitar un número variable de llamadas a herramientas dependiendo de la consulta del usuario; por ejemplo, obtener videos de tendencia y luego extraer los metadatos de cada video, o buscar videos sobre un tema y luego obtener las transcripciones de varios resultados.

Para manejar estos escenarios más complejos, crearás una cadena recursiva que pueda decidir dinámicamente cuántas llamadas a herramientas se necesitan y continuar el procesamiento hasta que se haya recopilado toda la información necesaria.

In [132]:
def ejecutar_herramienta_2(llamada_herramienta):
    """Ejecutar una única llamada a la herramienta y devolver ToolMessage."""
    try:
        resultado = mapeo_herramientas[llamada_herramienta["name"]].invoke(llamada_herramienta["args"])
        contenido = json.dumps(resultado) if isinstance(resultado, (dict, list)) else str(resultado)
    except Exception as e:
        contenido = f"Error: {str(e)}"
    
    return ToolMessage(
        content= contenido,
        tool_call_id= llamada_herramienta["id"]
    )

### *6.1. Definición de la Lógica de Procesamiento Principal*

Esta función gestiona la lógica de procesamiento principal de su cadena recursiva. Toma el historial de la conversación actual y:

1. Identifica el mensaje más reciente de la conversación.

2. Extrae todas las llamadas a herramientas de ese mensaje y las ejecuta en paralelo mediante la función auxiliar `ejecutar_herramienta_2`.

3. Actualiza el historial de mensajes añadiendo las respuestas de las herramientas.

4. Obtiene la siguiente respuesta del modelo de lenguaje en función de la conversación actualizada.

5. Devuelve el historial de mensajes completo y actualizado, incluyendo las respuestas de las herramientas y la nueva respuesta del modelo de lenguaje.

In [140]:
def proceso_llamada_herramienta(mensajes):
    """Procesador de llamadas recursivas a herramientas"""
    ultimo_mensaje = mensajes[-1]
    
    # Ejecutar todas las llamadas a herramientas en paralelo.
    mensajes_herramientas = [
        ejecutar_herramienta_2(tc) 
        for tc in getattr(ultimo_mensaje, 'tool_calls', [])
    ]
    
    # Agregar respuestas de la herramienta al historial de mensajes.
    actualizar_mensajes = mensajes + mensajes_herramientas
    
    # Obtener la siguiente respuesta del LLM utilizando el historial de mensajes actualizado.
    siguiente_respuesta_ia = llm_con_herramientas.invoke(actualizar_mensajes)
    
    return actualizar_mensajes + [siguiente_respuesta_ia]

### *6.2. Creación de la Condición de Parada Recursiva*

Esta función determina si el proceso recursivo debe continuar o finalizar. Realiza lo siguiente:

1. Analiza el historial de mensajes actual y examina el último mensaje.

2. Comprueba si ese mensaje contiene llamadas a herramientas mediante la función `getattr` (que gestiona de forma segura los casos en los que el atributo podría no existir).

3. Devuelve un valor booleano: `True` si hay más llamadas a herramientas que procesar y `False` cuando se alcanza un punto en el que el LLM ha proporcionado una respuesta final sin solicitar herramientas adicionales.

In [141]:
def deberia_continuar(mensajes):
    """Revisar si necesitas otra iteración"""
    ultimo_mensaje = mensajes[-1]
    return bool(getattr(ultimo_mensaje, 'tool_calls', None))

#### *6.3. Implementación de la Función Recursiva*

Esta función implementa la recursión que impulsa el proceso de llamada dinámica a herramientas:

1. Primero, verifica la condición de parada mediante la función `deberia_continuar` para determinar si se necesitan más llamadas a herramientas.

2. Si se necesitan más llamadas a herramientas, las procesa mediante la función `proceso_llamada_herramienta` y luego se llama recursivamente a sí misma con los mensajes actualizados.

3. Si no se necesitan más llamadas a herramientas, devuelve el historial final de mensajes, que contiene la conversación completa, incluyendo la respuesta final del LLM.

Tras definir esta función recursiva, la encapsulará en una `RunnableLambda` para que sea compatible con la arquitectura de cadena de LangChain.

In [151]:
def _cadena_recursiva(mensajes):
    """Procesar recursivamente las llamadas a las herramientas hasta su finalización."""
    if deberia_continuar(mensajes):
        nuevo_mensaje = proceso_llamada_herramienta(mensajes)
        return _cadena_recursiva(nuevo_mensaje)
    
    return mensajes

cadena_recursiva = RunnableLambda(_cadena_recursiva)

### *6.4. Creación de la Cadena Universal Completa*

Ahora, estás creando tu cadena universal final, capaz de gestionar cualquier tipo de consulta que requiera cualquier número de llamadas a herramientas. Esta cadena consta de tres pasos principales:

1. El primer paso convierte la consulta del usuario en un objeto `HumanMessage` con el formato adecuado.

2. El segundo paso envía este mensaje inicial a tu LLM equipado con herramientas y añade la primera respuesta del LLM al historial de mensajes.

3. El último paso transfiere la conversación a tu cadena recursiva, que gestionará todas las llamadas a herramientas posteriores hasta que el LLM proporcione una respuesta final.

Esta cadena universal es mucho más flexible que tu cadena anterior de pasos fijos, ya que puede adaptarse dinámicamente a consultas que requieren diferentes números y tipos de llamadas a herramientas.

In [152]:
cadena_universal = (
    RunnableLambda(lambda x: [HumanMessage(content= x["consulta"])])
    | RunnableLambda(lambda mensajes: mensajes + [llm_con_herramientas.invoke(mensajes)])
    | cadena_recursiva
)

In [153]:
print(cadena_universal.invoke({
    "consulta": "Muestra la metadata y minitatura del video de YouTube: https://www.youtube.com/watch?v=pbXxWaIwyg0"
})[-1])

content='### Metadata del Video\n\n- **Título:** Big Soto x Danny Ocean - CANAIMA (Visualizer)\n- **Vistas:** 366,408\n- **Duración:** 2 minutos y 32 segundos (152 segundos)\n- **Canal:** Big Soto\n- **Me gusta:** 6,740\n- **Comentarios:** 257\n- **Capítulos:** No disponibles\n\n### Miniaturas del Video\n\n1. ![Miniatura 1](https://i.ytimg.com/vi/pbXxWaIwyg0/3.jpg)\n2. ![Miniatura 2](https://i.ytimg.com/vi_webp/pbXxWaIwyg0/3.webp)\n3. ![Miniatura 3](https://i.ytimg.com/vi/pbXxWaIwyg0/2.jpg)\n4. ![Miniatura 4](https://i.ytimg.com/vi_webp/pbXxWaIwyg0/2.webp)\n5. ![Miniatura 5](https://i.ytimg.com/vi/pbXxWaIwyg0/1.jpg)\n6. ![Miniatura 6](https://i.ytimg.com/vi_webp/pbXxWaIwyg0/1.webp)\n7. ![Miniatura 7](https://i.ytimg.com/vi/pbXxWaIwyg0/mq3.jpg)\n8. ![Miniatura 8](https://i.ytimg.com/vi_webp/pbXxWaIwyg0/mq3.webp)\n9. ![Miniatura 9](https://i.ytimg.com/vi/pbXxWaIwyg0/hqdefault.jpg)\n10. ![Miniatura 10](https://i.ytimg.com/vi_webp/pbXxWaIwyg0/hqdefault.webp)\n\nSi necesitas más información

Copyright © IBM Corporation. All rights reserved.
